# 03 Preference Pair Audit

Purpose: generate task-specific preference pairs from the current SFT adapter, then inspect the emitted DPO dataset and audit trail directly from the Drive-backed runtime outputs.

This notebook assumes `notebooks/00_colab_setup.ipynb` has already mounted Drive and installed the Colab runtime dependencies from `json-ft-source`.


In [1]:
from pathlib import Path
import json

SOURCE_ROOT = Path('/content/drive/MyDrive/json-ft-source')
RUNTIME_ROOT = Path('/content/drive/MyDrive/json-ft-runs')
PROFILE = 'full'
RUN_NAME = 'pref-full-colab'
SFT_RUN_NAME = 'sft-full-colab'
PAIR_SOURCE_SAMPLE_PERCENT = 0.05
PAIR_SAMPLE_SEED = 17
PREFERENCE_BATCH_SIZE = 64
SOURCE_DATASET = SOURCE_ROOT / 'data' / 'manifests' / 'support_tickets_canonical.jsonl'
SFT_MANIFEST = SOURCE_ROOT / 'artifacts' / 'checkpoints' / f'{SFT_RUN_NAME}_adapter_manifest.json'

required_paths = [
    SOURCE_ROOT / 'scripts' / 'build_preference_pairs.py',
    SOURCE_ROOT / 'configs' / 'dpo.yaml',
    SOURCE_DATASET,
    SFT_MANIFEST,
]
missing = [path for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError('Missing required preference-build inputs:\n' + '\n'.join(str(path) for path in missing))

sft_manifest = json.loads(SFT_MANIFEST.read_text(encoding='utf-8'))
base_model = sft_manifest['base_model']
adapter_path = sft_manifest['adapter_path']

command = [
    'python',
    str(SOURCE_ROOT / 'scripts' / 'build_preference_pairs.py'),
    '--config',
    str(SOURCE_ROOT / 'configs' / 'dpo.yaml'),
    '--profile',
    PROFILE,
    '--run-name',
    RUN_NAME,
    '--runtime-root',
    str(RUNTIME_ROOT),
    '--input-path',
    str(SOURCE_DATASET),
    '--source-format',
    'json_extraction',
    '--source-split',
    'train',
    '--model-name-or-path',
    base_model,
    '--adapter-path',
    adapter_path,
    '--inference-batch-size',
    str(PREFERENCE_BATCH_SIZE),
    '--sample-percent',
    str(PAIR_SOURCE_SAMPLE_PERCENT),
    '--sample-seed',
    str(PAIR_SAMPLE_SEED),
    '--mirror-pairs-to-repo',
    '--mirror-audit-to-repo',
    '--mirror-summary-to-repo',
]

print("Running command:")
print(" ".join(command))


Running command:
python /content/drive/MyDrive/json-ft-source/scripts/build_preference_pairs.py --config /content/drive/MyDrive/json-ft-source/configs/dpo.yaml --profile full --run-name pref-full-colab --runtime-root /content/drive/MyDrive/json-ft-runs --input-path /content/drive/MyDrive/json-ft-source/data/manifests/support_tickets_canonical.jsonl --source-format json_extraction --source-split train --model-name-or-path Qwen/Qwen2.5-1.5B-Instruct --adapter-path /content/drive/MyDrive/json-ft-runs/persistent/checkpoints/sft/sft-full-colab/adapter --inference-batch-size 64 --sample-percent 0.05 --sample-seed 17 --mirror-pairs-to-repo --mirror-audit-to-repo --mirror-summary-to-repo


In [2]:
!python /content/drive/MyDrive/json-ft-source/scripts/build_preference_pairs.py \
    --config /content/drive/MyDrive/json-ft-source/configs/dpo.yaml \
    --profile {PROFILE} \
    --run-name {RUN_NAME} \
    --runtime-root /content/drive/MyDrive/json-ft-runs \
    --input-path {SOURCE_DATASET} \
    --source-format json_extraction \
    --source-split train \
    --model-name-or-path {base_model} \
    --adapter-path {adapter_path} \
    --inference-batch-size {PREFERENCE_BATCH_SIZE} \
    --sample-percent {PAIR_SOURCE_SAMPLE_PERCENT} \
    --sample-seed {PAIR_SAMPLE_SEED} \
    --mirror-pairs-to-repo \
    --mirror-audit-to-repo \
    --mirror-summary-to-repo


Preference pair generation
Config: /content/drive/MyDrive/json-ft-source/configs/dpo.yaml
Profile: full
Run name: pref-full-colab
Input path: /content/drive/MyDrive/json-ft-source/data/manifests/support_tickets_canonical.jsonl
Source split: train
Model name or path: Qwen/Qwen2.5-1.5B-Instruct
Adapter path: /content/drive/MyDrive/json-ft-runs/persistent/checkpoints/sft/sft-full-colab/adapter
Prompt source: messages
Dataset build summary: /content/drive/MyDrive/json-ft-source/data/manifests/support_tickets_dataset_build_summary.json
Dataset composition summary: /content/drive/MyDrive/json-ft-source/artifacts/metrics/support_tickets_dataset_composition.json
Quality gates: {'minimum_score_gap': 0.4, 'max_similarity_ratio': 0.96, 'reject_same_failure_mode': True, 'require_chosen_schema_valid': True, 'minimum_candidate_count_after_dedup': 2}
Source rows: 3558
Source subset: {'original_row_count': 71161, 'selected_row_count': 3558, 'selected_fraction': 0.0499992973679403, 'sample_mode': 'dete

In [3]:
from pathlib import Path
import json

run_dir = RUNTIME_ROOT / 'persistent' / 'preferences' / RUN_NAME
pairs_path = run_dir / f'{RUN_NAME}_dpo_pairs.jsonl'
audit_path = run_dir / f'{RUN_NAME}_preference_audit.jsonl'
summary_path = run_dir / f'{RUN_NAME}_preference_summary.json'
diagnostics_path = run_dir / f'{RUN_NAME}_preference_diagnostics.json'

summary = json.loads(summary_path.read_text(encoding='utf-8'))
diagnostics = json.loads(diagnostics_path.read_text(encoding='utf-8'))
audit_rows = [json.loads(line) for line in audit_path.read_text(encoding='utf-8').splitlines() if line.strip()]
pair_rows = [json.loads(line) for line in pairs_path.read_text(encoding='utf-8').splitlines() if line.strip()]

print(json.dumps(summary, indent=2, sort_keys=True))
print(json.dumps(diagnostics, indent=2, sort_keys=True))
print(f'Loaded {len(pair_rows)} DPO pairs and {len(audit_rows)} audit rows from {run_dir}')

skipped_rows = [row for row in audit_rows if row.get("skip_reason")]
emitted_rows = [row for row in audit_rows if not row.get("skip_reason")]

print("Skipped rows by reason:")
for reason, count in sorted(summary.get("skipped_counts", {}).items()):
    print(f"- {reason}: {count}")

if pair_rows:
    print("\nExample emitted DPO pair:")
    print(json.dumps(pair_rows[0], indent=2, sort_keys=True))

if emitted_rows:
    example = emitted_rows[0]
    print("\nExample emitted audit row")
    print(f"record_id: {example['record_id']}")
    print(f"decision_rationale: {example['decision_rationale']}")
    print(f"chosen_index: {example['chosen_index']}")
    print(f"rejected_index: {example['rejected_index']}")
    print("\nTop-ranked candidate scorecard:")
    print(json.dumps(example['candidates'][0]['scorecard'], indent=2, sort_keys=True))
    print("\nLowest-ranked candidate scorecard:")
    print(json.dumps(example['candidates'][-1]['scorecard'], indent=2, sort_keys=True))

if skipped_rows:
    skipped = skipped_rows[0]
    print("\nExample skipped row")
    print(f"record_id: {skipped['record_id']}")
    print(f"skip_reason: {skipped['skip_reason']}")
    print(f"decision_rationale: {skipped['decision_rationale']}")


{
  "adapter_path": "/content/drive/MyDrive/json-ft-runs/persistent/checkpoints/sft/sft-full-colab/adapter",
  "average_chosen_vs_rejected_score_gap": {
    "actions_f1_gap": 0.0,
    "concision_gap": 0.0,
    "hallucinated_key_reduction": 0.0,
    "null_handling_gap": 0.1550034989503149,
    "numeric_score_gap": 8.967697259839065,
    "parses_json_gap": 0.0003498950314905528,
    "schema_valid_gap": 0.940517844646606,
    "structured_field_match_gap": 1.7830650804758572,
    "summary_faithfulness_gap": 0.9041704615395489,
    "summary_word_reduction": -6.503498950314905
  },
  "build_summary_path": "/content/drive/MyDrive/json-ft-source/data/manifests/support_tickets_dataset_build_summary.json",
  "candidate_count_total": 28464,
  "candidate_json_valid_rate": 0.9976461495222035,
  "candidate_schema_pass_rate": 0.7062956717256886,
  "chosen_schema_valid_rate": 1.0,
  "composition_summary_path": "/content/drive/MyDrive/json-ft-source/artifacts/metrics/support_tickets_dataset_composition

## Manual Spot-Check Checklist

- Confirm the emitted pair count is non-zero and the skip reasons are understandable.
- For at least 5 emitted rows, read the input text, gold payload, chosen candidate, and rejected candidate together.
- Verify the chosen response is schema-valid, has no hallucinated keys, and is more correct than rejected on the task labels.
- Verify the rejected response is genuinely worse, not just differently phrased.
- Check that summary differences reflect faithfulness and concision, not hidden semantic loss.
- Inspect at least 2 skipped rows and confirm the skip reason is appropriate before trusting the emitted-pair rate.
- Do not start DPO training until the audit sample looks clean enough to defend in `docs/05_failure_cases.md`.
